# Image Recognition in Snowflake using Snowpark Python and PyTorch

*NOTE: For prerequisites, step-by-step guide, and instructions for running this application, please refer to the [QuickStart Guide](https://quickstarts.snowflake.com/guide/image_recognition_snowpark_pytorch_streamlit_openai/?_fsi=THrZMtDg,%20THrZMtDg#0).*


In [ ]:
# Snowpark
from snowflake.snowpark.functions import udf
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Misc
import pandas as pd
import json
import cachetools
import logging
logger = logging.getLogger("snowflake.snowpark.session")
logger.setLevel(logging.ERROR)

session = get_active_session()

## Setup

Create a Snowflake table and internal stage by running the following commands. The table will store the image data and the stage is for storing serialized Snowpark Python UDF code.

In [ ]:
create table if not exists images (file_name string, image_bytes string);

In [ ]:
create stage if not exists dash_files;

## PyTorch and Snowpark Python

For this particular application, we will be using [PyTorch implementation of MobileNet V3](https://github.com/d-li14/mobilenetv3.pytorch). *Note: A huge thank you to the [authors](https://github.com/d-li14/mobilenetv3.pytorch?_fsi=THrZMtDg,%20THrZMtDg&_fsi=THrZMtDg,%20THrZMtDg#citation) for the research and making the pre-trained models available under [MIT License](https://github.com/d-li14/mobilenetv3.pytorch/blob/master/LICENSE).*

1) Once you have dowloaded the pre-trained model files from the repo link above, upload them onto Snowflake (internal) stage `dash_files` using [Snowsight](https://docs.snowflake.com/en/user-guide/data-load-local-file-system-stage-ui#upload-files-onto-a-named-internal-stage) so that they can be added as dependencies on the Snowpark for Python UDF for inference.
2) Snowpark Python User-Defined Function (UDF) for image recognition

    To deploy the pre-trained model for inference, let's **create and register a Snowpark Python UDF and add the model files as dependencies**. Once registered, getting new predictions is as simple as calling the function by passing in data.*NOTE: Scalar UDFs operate on a single row / set of data points and are great for online inference in real-time. And this UDF is called from [Streamlit apps](https://quickstarts.snowflake.com/guide/image_recognition_snowpark_pytorch_streamlit_openai/?_fsi=THrZMtDg,%20THrZMtDg#1).*

    TIP: For more information on Snowpark Python User-Defined Functions, refer to the [docs](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-udfs.html).


In [ ]:
session.clear_packages()
session.clear_imports()

# Add model files and test images as dependencies on the UDF
session.add_import('@dash_files/imagenet1000_clsidx_to_labels.txt')
session.add_import('@dash_files/mobilenetv3.py')
session.add_import('@dash_files/mobilenetv3-large-1cd25616.pth')

# Add Python packages from Snowflake Anaconda channel
session.add_packages('snowflake-snowpark-python','torchvision','joblib','cachetools')

@cachetools.cached(cache={})
def load_class_mapping(filename):
  with open(filename, "r") as f:
   return f.read()

@cachetools.cached(cache={})
def load_model():
  import sys
  import torch
  from torchvision import models, transforms
  import ast
  from mobilenetv3 import mobilenetv3_large

  IMPORT_DIRECTORY_NAME = "snowflake_import_directory"
  import_dir = sys._xoptions[IMPORT_DIRECTORY_NAME]

  model_file = import_dir + 'mobilenetv3-large-1cd25616.pth'
  imgnet_class_mapping_file = import_dir + 'imagenet1000_clsidx_to_labels.txt'

  IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

  transform = transforms.Compose([
      transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
  ])

  # Load the Imagenet {class: label} mapping
  cls_idx = load_class_mapping(imgnet_class_mapping_file)
  cls_idx = ast.literal_eval(cls_idx)

  # Load pretrained image recognition model
  model = mobilenetv3_large()
  model.load_state_dict(torch.load(model_file))

  # Configure pretrained model for inference
  model.eval().requires_grad_(False)

  return model, transform, cls_idx

@udf(name='image_recognition_using_bytes',session=session,replace=True,is_permanent=True,stage_location='@dash_files')
def image_recognition_using_bytes(image_bytes_in_str: str) -> str:
  from io import BytesIO
  import torch
  from PIL import Image

  image_bytes = bytes.fromhex(image_bytes_in_str)

  model, transform, cls_idx = load_model()
  img = Image.open(BytesIO(image_bytes)).convert('RGB')
  img = transform(img).unsqueeze(0)

  # Get model output and human text prediction
  logits = model(img)

  outp = torch.nn.functional.softmax(logits, dim=1)
  _, idx = torch.topk(outp, 1)
  idx.squeeze_()
  predicted_label = cls_idx[idx.item()]

  return f"{predicted_label}"

*NOTE: To build and run Streamlit applications that will use the above UDF, continue and complete the steps outlined in the [QuickStart Guide](https://quickstarts.snowflake.com/guide/image_recognition_snowpark_pytorch_streamlit_openai/?_fsi=THrZMtDg,%20THrZMtDg#4).*